In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn transformers torch evaluate datasets accelerate peft')
    os.system('pip uninstall -y torchvision')
    print("Setup complete!")


In [2]:
import os
import json
import pandas as pd
import torch
import evaluate
import numpy as np
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from peft import get_peft_model, LoraConfig, TaskType
import torch.distributed.tensor

if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

model_name = "papluca/xlm-roberta-base-language-detection"
batch_size = 4
learning_rate = 2e-5
num_epochs = 10

TARGET_LANGUAGES = {
    "eng": "en", "sin": "si", "san": "sa", "tam": "ta", "hin": "hi", "ben": "bn", "arb": "ar", "fra": "fr", "deu": "de", "pli": "pi",
    "jpn": "ja", "nld": "nl", "pol": "pl", "ita": "it", "por": "pt", "tur": "tr", "spa": "es", "ell": "el", "urd": "ur", "bul": "bg", "cmn": "zh", "rus": "ru", "tha": "th", "swh": "sw", "vie": "vi"
}

print("Loading original model configuration to determine label mappings...")
config = AutoConfig.from_pretrained(model_name)

for old_code, new_code in TARGET_LANGUAGES.items():
    if new_code not in config.label2id:
        idx = len(config.label2id)
        config.label2id[new_code] = idx
        config.id2label[idx] = new_code

tokenizer = AutoTokenizer.from_pretrained(model_name)
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="micro")


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading original model configuration to determine label mappings...


In [3]:
def train_experiment(train_path, val_path, save_name):
    print(f"\n{'='*60}")
    print(f"STARTING EXPERIMENT: {save_name}")
    print(f"{'='*60}")
    
    output_model_dir = f'models/finetuned/xlmr/{save_name}'
    
    def load_data(jsonl_path):
        records = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                rec = json.loads(line)
                mapped = TARGET_LANGUAGES.get(rec['label'], rec['label'])
                if mapped in config.label2id:
                    records.append({"text": rec["text"], "label": config.label2id[mapped]})
        return pd.DataFrame(records)

    train_df = load_data(train_path)
    val_df = load_data(val_path)
    
    print(f"Loaded {len(train_df)} training samples and {len(val_df)} validation samples.")
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

    train_dataset = HFDataset.from_pandas(train_df)
    val_dataset = HFDataset.from_pandas(val_df)
    
    tokenized_train = train_dataset.map(tokenize_function, batched=True).remove_columns(["text"])
    tokenized_val = val_dataset.map(tokenize_function, batched=True).remove_columns(["text"])
    tokenized_train.set_format("torch")
    tokenized_val.set_format("torch")
    
    print("Loading fresh model and expanding classification head...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    old_out_features = model.classifier.out_proj.out_features
    new_out_features = len(config.label2id)
    
    if new_out_features > old_out_features:
        new_out_proj = torch.nn.Linear(model.classifier.out_proj.in_features, new_out_features)
        new_out_proj.weight.data[:old_out_features] = model.classifier.out_proj.weight.data
        new_out_proj.bias.data[:old_out_features] = model.classifier.out_proj.bias.data
        torch.nn.init.xavier_uniform_(new_out_proj.weight.data[old_out_features:])
        torch.nn.init.zeros_(new_out_proj.bias.data[old_out_features:])
        model.classifier.out_proj = new_out_proj
        model.num_labels = new_out_features
        model.config = config
        
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=256,
        lora_alpha=512,
        lora_dropout=0.1,
        target_modules=["query", "key", "value", "dense"], 
        modules_to_save=["classifier"]
    )
    model = get_peft_model(model, lora_config)
    
    training_args = TrainingArguments(
        output_dir=output_model_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=16 // batch_size,
        fp16=torch.cuda.is_available(),
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print("Starting Fine-tuning...")
    trainer.train()
    
    print(f"Saving final model to {output_model_dir}...")
    config.save_pretrained(output_model_dir)
    trainer.save_model(output_model_dir)
    tokenizer.save_pretrained(output_model_dir)
    print("Finetuning Complete!\n")


## Experiment 1: No Rehearsal
We train exclusively on our custom Sinhala-script data to observe Catastrophic Forgetting in large language models.

In [4]:
train_experiment(
    train_path="datasets/finetuning/train.jsonl",
    val_path="datasets/finetuning/val_mixed.jsonl",
    save_name="xlmr_no_rehearsal"
)



STARTING EXPERIMENT: xlmr_no_rehearsal
Loaded 60285 training samples and 8986 validation samples.


Map: 100%|██████████| 8986/8986 [00:00<00:00, 12355.51 examples/s]


Loading fresh model and expanding classification head...
Starting Fine-tuning...


  1%|▏         | 500/37680 [02:30<3:05:52,  3.33it/s]

{'loss': 0.1714, 'grad_norm': 0.005028851330280304, 'learning_rate': 1.9736730360934185e-05, 'epoch': 0.13}


...

{'loss': 0.0108, 'grad_norm': 0.02761632204055786, 'learning_rate': 1.204299363057325e-05, 'epoch': 3.98}


                                                         
 40%|████      | 15072/37680 [1:20:14<1:47:14,  3.51it/s]/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /papluca/xlm-roberta-base-language-detection/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: d51ec6f9-9b75-448e-b84e-7b326a689114)') - silently ignoring the lookup for the file config.json in papluca/xlm-roberta-base-language-detection.
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for

{'eval_loss': 4.636631965637207, 'eval_f1': 0.7755397284665034, 'eval_runtime': 58.0266, 'eval_samples_per_second': 154.86, 'eval_steps_per_second': 38.724, 'epoch': 4.0}


 40%|████      | 15072/37680 [1:20:15<2:00:22,  3.13it/s]
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /papluca/xlm-roberta-base-language-detection/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 74b88fec-483f-4d53-a3a5-1fb18421f827)') - silently ignoring the lookup for the file config.json in papluca/xlm-roberta-base-language-detection.
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark

{'train_runtime': 4815.1447, 'train_samples_per_second': 125.199, 'train_steps_per_second': 7.825, 'train_loss': 0.021789170991463266, 'epoch': 4.0}
Saving final model to models/finetuned/xlmr/xlmr_no_rehearsal...
Finetuning Complete!



## Experiment 2: With Rehearsal (Mixed Dataset)
We train on the mixed dataset (Aya + our custom data) to retain structural boundaries for the existing baseline languages.

In [5]:
train_experiment(
    train_path="datasets/finetuning/train_mixed.jsonl",
    val_path="datasets/finetuning/val_mixed.jsonl",
    save_name="xlmr_with_rehearsal"
)



STARTING EXPERIMENT: xlmr_with_rehearsal
Loaded 108218 training samples and 8986 validation samples.


Map: 100%|██████████| 8986/8986 [00:00<00:00, 9981.19 examples/s] 


Loading fresh model and expanding classification head...
Starting Fine-tuning...


  1%|          | 500/67630 [02:29<5:34:58,  3.34it/s]

{'loss': 0.1951, 'grad_norm': 17.604808807373047, 'learning_rate': 1.9853319532751737e-05, 'epoch': 0.07}


...

 60%|██████    | 40581/67630 [3:26:45<2:12:37,  3.40it/s]/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /papluca/xlm-roberta-base-language-detection/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: c9cdde5f-c757-46ec-a266-36bd7062ccb2)') - silently ignoring the lookup for the file config.json in papluca/xlm-roberta-base-language-detection.
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/

{'eval_loss': 0.02476327121257782, 'eval_f1': 0.9974404629423548, 'eval_runtime': 58.341, 'eval_samples_per_second': 154.025, 'eval_steps_per_second': 38.515, 'epoch': 6.0}


 60%|██████    | 40581/67630 [3:26:45<2:17:48,  3.27it/s]

{'train_runtime': 12405.6351, 'train_samples_per_second': 87.233, 'train_steps_per_second': 5.452, 'train_loss': 0.017514927341235293, 'epoch': 6.0}
Saving final model to models/finetuned/xlmr/xlmr_with_rehearsal...
Finetuning Complete!




/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /papluca/xlm-roberta-base-language-detection/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 388ce255-5c86-4936-98b0-22339f6adcc8)') - silently ignoring the lookup for the file config.json in papluca/xlm-roberta-base-language-detection.
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/sav